# Positional Encoding, Visually

This notebook is a companion to `docs/2026-09-10/pe-refactor-notes.md` and the experiment results in `docs/2026-09-10/`. Where those documents report *numbers* (perplexity, accuracy), this notebook shows the actual *objects* each positional-encoding (PE) scheme computes — the tables, rotations, and bias matrices a model built from `src/models/layers/positional_encoding/` actually uses at each attention layer.

Run this after `scripts/run_pe_experiments.py` has produced checkpoints under `checkpoints/experiments/` — a couple of sections load a trained checkpoint to show what was actually *learned*, not just what was initialized.

## A worked lesson: the bug that started this notebook

Earlier revisions of this codebase applied RoPE and Shaw et al.'s relative encoding **once, at the embedding layer** — before the first attention layer's Q/K projection. That is a very common mistake, and it *runs without error*, which is what makes it dangerous: the model trains, the loss goes down, nothing looks wrong.

The problem: RoPE's entire point is that `q_i · k_j` (after rotation) depends only on `i - j`, not on `i` and `j` separately. Rotating the *embedding* once, before `W_Q`/`W_K` are applied, does not preserve that property — the linear projections mix the rotated dimensions arbitrarily, and by layer 2 there is no relative-position information left at all. The fix moves the rotation **inside every attention layer**, applied directly to `q` and `k` after projection (see `src/models/layers/attentions.py`). Section 3 below shows the property this actually buys you; the crash test in `docs/2026-09-10/length-generalization.md` shows what the *old* (embedding-level) version of the analogous absolute/learned encodings couldn't do at all: run on a sequence longer than they were trained on.

**Lesson for anyone implementing a paper's positional encoding**: ask "where does the paper's equation live — added to the token embedding, or inside the attention score?" *before* writing code. Both integrate cleanly into a forward pass. Only one of them is what the paper means.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent  # running from notebooks/
sys.path.insert(0, str(ROOT))

import glob
import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline

from src.models.layers.positional_encoding.sinusoidal import SinusoidalPositionalEncoding
from src.models.layers.positional_encoding.learnable import LearnablePositionalEncoding
from src.models.layers.positional_encoding.rotary import RotaryPositionalEncoding
from src.models.layers.positional_encoding.alibi import ALiBiPositionalEncoding
from src.models.layers.positional_encoding.relative import RelativePositionalEncoding
from src.models.layers.positional_encoding.t5_relative import get_relative_positions
from src.models.configs import BertConfig
from src.models.model import BertMLM
from src.models.tokenizer import BertTokenizer
from src.utils import read_yaml

plt.rcParams["figure.dpi"] = 110
SEQ_LEN = 32
D_MODEL = 32

## 1. Absolute / sinusoidal PE — a fixed table, added once

`SinusoidalPositionalEncoding` (Vaswani et al., 2017) precomputes a `(max_len, d_model)` table of sines and cosines and adds it directly to the token embedding, once, before the first attention layer. It never changes during training and never depends on content — every sequence gets the exact same table.

In [ ]:
pe = SinusoidalPositionalEncoding(d_model=D_MODEL, max_len=SEQ_LEN)
table = pe.pe[0, :SEQ_LEN].numpy()

fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(table.T, aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xlabel("position")
ax.set_ylabel("embedding dimension")
ax.set_title("Sinusoidal PE table: sin/cos at every (position, dimension)")
fig.colorbar(im, ax=ax, label="value")
plt.tight_layout()
plt.show()

## 2. Learnable PE — what does training actually shape it into?

`LearnablePositionalEncoding` starts as a small random table (`Uniform(-0.02, 0.02)`) and is updated by gradient descent like any other parameter. Below: the table *before* training (tiny, structureless noise) next to the table pulled from an actual trained checkpoint (`checkpoints/experiments/learnable/`, from `scripts/run_pe_experiments.py`'s MLM comparison).

In [ ]:
fresh = LearnablePositionalEncoding(d_model=256, max_len=64, init_std=0.0)
fresh_table = fresh.pe.detach().numpy()

ckpt_paths = sorted(glob.glob(str(ROOT / "checkpoints/experiments/learnable/*/checkpoint-final/model.pt")))
if ckpt_paths:
    state_dict = torch.load(ckpt_paths[-1], map_location="cpu")
    trained_table = state_dict["bert.embeddings.position_embeddings.pe"].numpy()

    vmax = np.abs(trained_table).max()
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
    for ax, table, title in zip(
        axes, [fresh_table, trained_table], ["before training (init)", "after MLM training"]
    ):
        im = ax.imshow(table.T, aspect="auto", cmap="RdBu_r", vmin=-vmax, vmax=vmax)
        ax.set_title(title)
        ax.set_xlabel("position")
    axes[0].set_ylabel("embedding dimension")
    fig.colorbar(im, ax=axes.tolist(), label="value", shrink=0.8)
    plt.suptitle("Learnable PE: random init vs. what 100 steps of MLM training shaped it into")
    plt.show()
else:
    print("No trained checkpoint found - run scripts/run_pe_experiments.py first.")

## 3. RoPE — no table at all, a rotation applied inside attention

RoPE (Su et al., 2021) doesn't add anything to the embedding. Instead, every attention layer rotates each query/key vector by an angle proportional to its position, in 2D sub-planes across the head dimension. The property that matters: **the dot product of a rotated query at position `i` and a rotated key at position `j` depends only on `i - j`.**

The plot below is the actual property check from `tests/models/layers/positional_encoding/test_rotary.py`, made visual: the same pair of vectors, placed at increasing absolute positions but always 5 positions apart, always producing the same dot product.

In [ ]:
rope = RotaryPositionalEncoding(d_model=D_MODEL, max_len=64)
torch.manual_seed(0)
base_q = torch.randn(1, 1, 1, D_MODEL)
base_k = torch.randn(1, 1, 1, D_MODEL)
GAP = 5


def score_at(start_pos: int) -> float:
    seq_len = start_pos + GAP + 1
    q = torch.zeros(1, 1, seq_len, D_MODEL)
    k = torch.zeros(1, 1, seq_len, D_MODEL)
    q[0, 0, start_pos] = base_q
    k[0, 0, start_pos + GAP] = base_k
    q_rot, k_rot = rope.rotate_qk(q, k)
    return (q_rot[0, 0, start_pos] @ k_rot[0, 0, start_pos + GAP]).item()


starts = list(range(0, 40, 2))
scores = [score_at(s) for s in starts]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(starts, scores, "o-", color="#2a78d6")
ax.set_xlabel("absolute start position of q (k is always 5 positions later)")
ax.set_ylabel("q · k after rotation")
ax.set_title("RoPE: dot product depends only on relative offset (flat line), not absolute position")
ax.set_ylim(min(scores) - 0.5, max(scores) + 0.5)
plt.tight_layout()
plt.show()
print(f"variance across all {len(scores)} absolute positions: {np.var(scores):.2e} (should be ~0)")

## 4. ALiBi — a fixed, per-head penalty for distance

ALiBi (Press et al., 2021) adds no rotation and no learned table either. It adds a static bias directly to the attention *scores*: `bias = -slope_h * |i - j|`. Every head gets its own slope, in a geometric sequence, so some heads attend broadly and others stay strictly local — fixed at initialization, never learned, never touching the embeddings.

In [ ]:
alibi = ALiBiPositionalEncoding(num_heads=4)
bias = alibi.attention_bias(seq_len=SEQ_LEN, device=torch.device("cpu"), dtype=torch.float32)[0]

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5), sharey=True)
for h, ax in enumerate(axes):
    im = ax.imshow(bias[h].numpy(), cmap="Blues_r", aspect="auto")
    ax.set_title(f"head {h}\nslope={alibi.slopes[h]:.3f}")
    ax.set_xlabel("key position")
axes[0].set_ylabel("query position")
fig.colorbar(im, ax=axes.tolist(), label="bias added to attention score", shrink=0.8)
plt.suptitle("ALiBi bias per head: steeper slope -> stronger penalty for distant tokens")
plt.show()

## 5. Relative (Shaw et al.) — a learned table indexed by *offset*, not position

Shaw et al. (2018) learn a small table of embeddings, one per possible relative offset `i - j`, clipped to `±clipping_distance`. Unlike the sinusoidal/learnable PE above, this table is indexed by *distance between tokens*, so the same offset always gets the same vector no matter where in the sequence it occurs — and it's added into the attention score (`q_i · a_ij`), not the embedding.

In [ ]:
rel = RelativePositionalEncoding(d_model=D_MODEL, max_len=64, clipping_distance=4)
rel_idx = rel._rel_indices(seq_len=SEQ_LEN, device=torch.device("cpu")).numpy()

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(rel_idx, cmap="viridis", aspect="auto")
ax.set_xlabel("key position")
ax.set_ylabel("query position")
ax.set_title("Relative PE bucket index (offsets beyond ±4 all clip to the same bucket)")
fig.colorbar(im, ax=ax, label="bucket index into the learned offset table")
plt.tight_layout()
plt.show()

## 6. T5-style bucketed relative — log-scale, not linear

T5's relative bias (Raffel et al., 2019) also buckets by offset, but on a *log* scale: nearby offsets each get their own bucket, while distant offsets are grouped together increasingly coarsely. This is the bonus scheme wired up during the correctness fix — the bucketing math (`_get_relative_position_bucket`) already existed in this codebase but was never connected to `get_pos_encoder`.

In [ ]:
buckets = get_relative_positions(seq_len=SEQ_LEN, bidirectional=True, num_buckets=32, max_distance=128).numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
im0 = axes[0].imshow(buckets, cmap="viridis", aspect="auto")
axes[0].set_title("T5 bucket index per (query, key) pair")
axes[0].set_xlabel("key position")
axes[0].set_ylabel("query position")
fig.colorbar(im0, ax=axes[0])

q_pos = SEQ_LEN // 2
key_positions = np.arange(SEQ_LEN)
offsets = key_positions - q_pos
row = buckets[q_pos, :]
axes[1].plot(offsets, row, color="#eb6834")
axes[1].set_xlabel("relative offset (key - query)")
axes[1].set_ylabel("bucket index")
axes[1].set_title("Bucket width grows with distance (log-scale binning)")
plt.tight_layout()
plt.show()

## 7. What this actually does to attention

Numbers and bucket diagrams are one level removed from the thing that matters: what does the trained model actually attend to? Below, we load two real checkpoints from the MLM comparison (`absolute` and `relative`) and capture their first layer's real attention weights on the same sentence, via a forward hook (no model code changes needed — `torch.nn.Module.register_forward_hook` on the dropout layer right after softmax gives us the post-softmax attention matrix for free).

In [ ]:
def load_trained_mlm(pe_type: str) -> BertMLM:
    config = read_yaml(str(ROOT / "configs/experiment.yaml"))
    mc = config["model"]
    bert_config = BertConfig(
        vocab_size=mc["vocab_size"], hidden_size=mc["hidden_size"],
        num_hidden_layers=mc["num_hidden_layers"], num_attention_heads=mc["num_attention_heads"],
        intermediate_size=mc["intermediate_size"], max_position_embeddings=mc["max_position_embeddings"],
        position_embedding_type=pe_type,
    )
    model = BertMLM(bert_config)
    ckpts = sorted(glob.glob(str(ROOT / f"checkpoints/experiments/{pe_type}/*/checkpoint-final/model.pt")))
    if not ckpts:
        raise FileNotFoundError(f"No checkpoint for {pe_type} - run scripts/run_pe_experiments.py first.")
    model.load_state_dict(torch.load(ckpts[-1], map_location="cpu"))
    model.eval()
    return model


def capture_attention(model: BertMLM, input_ids, attention_mask, layer: int = 0):
    captured = {}

    def hook(module, inp, out):
        captured["att"] = inp[0].detach()

    handle = model.bert.encoder[layer].attention.dropout_attn.register_forward_hook(hook)
    with torch.no_grad():
        model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=torch.zeros_like(input_ids),
        )
    handle.remove()
    return captured["att"][0]  # drop batch dim -> (num_heads, T, T)


tokenizer = BertTokenizer(model_id="bert-base-uncased")
sentence = "the cat sat on the mat"
enc = tokenizer.encode(sentence, max_length=16, truncation=True, padding="max_length", return_tensors="pt")
tokens = ["[CLS]"] + tokenizer.tokenize(sentence) + ["[SEP]"]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, pe_type in zip(axes, ["absolute", "relative"]):
    model = load_trained_mlm(pe_type)
    att = capture_attention(model, enc["input_ids"], enc["attention_mask"])
    head0 = att[0, : len(tokens), : len(tokens)].numpy()
    im = ax.imshow(head0, cmap="viridis")
    ax.set_xticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=90)
    ax.set_yticks(range(len(tokens)))
    ax.set_yticklabels(tokens)
    ax.set_title(f"{pe_type} - layer 0, head 0")
plt.suptitle(f'Attention on: "{sentence}"')
plt.tight_layout()
plt.show()

## 8. The zero point: no positional encoding at all\n\nEvery section above shows a *mechanism*. This section shows what happens with none - `NoPositionalEncoding` (`src/models/layers/positional_encoding/identity.py`) adds nothing anywhere, at any layer. It's the scientific control every scheme above should be read relative to: without it, \"scheme A beats scheme B\" has no zero point to compare against.\n\nFor a *bidirectional* encoder like this project's BERT (full self-attention, no causal mask), removing all positional signal has one precise, checkable consequence: the whole model becomes **permutation-equivariant** - shuffle the input tokens and every per-token output shuffles identically. Below, that property is demonstrated directly (this is exactly `tests/models/layers/positional_encoding/test_identity.py`, without the `assert`): the same sentence, once in its original order and once with a random shuffle, then the shuffled run's output un-shuffled back - the two match to floating-point precision, model logits included.

In [ ]:
from src.models.layers.positional_encoding import NoPositionalEncoding\n\nconfig = BertConfig(\n    vocab_size=64, hidden_size=32, num_hidden_layers=2, num_attention_heads=4,\n    intermediate_size=64, max_position_embeddings=32, position_embedding_type=\"none\",\n)\nmodel = BertMLM(config)\nmodel.eval()\n\ntorch.manual_seed(1)\ninput_ids = torch.randint(0, config.vocab_size, (1, 10))\nattention_mask = torch.ones(1, 10, dtype=torch.long)\ntoken_type_ids = torch.zeros(1, 10, dtype=torch.long)\n\nperm = torch.randperm(10)\ninv_perm = torch.argsort(perm)\n\nwith torch.no_grad():\n    logits = model(input_ids, attention_mask, token_type_ids)\n    logits_on_shuffled = model(input_ids[:, perm], attention_mask, token_type_ids)\n\nunshuffled_back = logits_on_shuffled[:, inv_perm]\nmax_diff = (unshuffled_back - logits).abs().max().item()\n\nfig, axes = plt.subplots(1, 2, figsize=(11, 3.5))\naxes[0].imshow(logits[0, :, :20].numpy(), aspect=\"auto\", cmap=\"RdBu_r\")\naxes[0].set_title(\"logits: original order\")\naxes[0].set_xlabel(\"vocab (first 20 dims)\"); axes[0].set_ylabel(\"sequence position\")\naxes[1].imshow(unshuffled_back[0, :, :20].numpy(), aspect=\"auto\", cmap=\"RdBu_r\")\naxes[1].set_title(\"logits: shuffled input, then un-shuffled output\")\naxes[1].set_xlabel(\"vocab (first 20 dims)\")\nplt.suptitle(f\"NoPE is permutation-equivariant - max difference between the two panels: {max_diff:.2e}\")\nplt.tight_layout()\nplt.show()\nprint(f\"max |difference|: {max_diff:.2e} (should be ~0 - floating-point only)\")

**Caveat**: these checkpoints were trained for only 100 steps on a 2,000-row subset (see `docs/2026-09-10/experiment-design.md`) — a feasibility check, not a converged model. Don't over-read specific attention patterns here; the point of this section is the *method* (how to pull real attention weights out of an unmodified model via a forward hook), which works the same way on a fully-trained model.

## Summary

| Scheme | Where it lives | What it is |
|---|---|---|
| Absolute / sinusoidal | embedding, added once | fixed sin/cos table |
| Learnable | embedding, added once | trainable table |
| RoPE | inside every attention layer | rotation of q/k, indexed by absolute position, invariant to relative offset |
| Relative (Shaw et al.) | inside every attention layer | learned table indexed by *clipped offset* |
| ALiBi | inside every attention layer | fixed per-head linear penalty on distance |
| T5-style bucketed | inside every attention layer | learned table indexed by *log-scale bucketed offset* |

See `src/models/layers/positional_encoding/` for the implementations, `tests/models/layers/positional_encoding/` for the correctness property each one is checked against, and `docs/2026-09-10/pe-refactor-notes.md` for the full story of what was wrong before and why.